# 05 - LOSO Two-Tower Neural Fusion (Physio + EEG)

This notebook trains a two-tower neural model on the built fusion dataset:

1. Physio tower (ECG/EDA/EMG/RESP features)
2. EEG tower (`eeg_features_5s__*` features)
3. Gated fusion classifier
4. LOSO fold training on Colab T4
5. Export fold and aggregate metrics

In [1]:
# SECTION 1: Imports
import json
from dataclasses import dataclass, asdict
from pathlib import Path

import numpy as np
import pandas as pd

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import f1_score, balanced_accuracy_score, accuracy_score

print('✓ Imports loaded')
print('CUDA:', torch.cuda.is_available())

✓ Imports loaded
CUDA: True


In [2]:
# SECTION 2: Config and Paths
@dataclass
class NNConfig:
    output_version: str = 'v1_loso_two_tower'
    batch_size: int = 256
    lr: float = 2e-3
    weight_decay: float = 1e-4
    max_epochs: int = 30
    patience: int = 8
    hidden_dim: int = 256
    dropout: float = 0.2
    max_folds: int = 5  # Dev mode: evaluate 5 representative LOSO folds
    random_state: int = 42
    fold_selection_mode: str = 'fusion_representative'  # fusion_representative | first_n

CFG = NNConfig()

try:
    from google.colab import drive
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    try:
        drive.mount('/content/drive', force_remount=False)
    except Exception:
        pass
    ROOT = Path('/content/drive/MyDrive')
else:
    ROOT = Path.home() / 'Desktop' / 'thesis'

FUSION_PATH = ROOT / 'research_outputs' / 'fusion' / 'v1_fusion' / 'fusion_dataset.csv'
OUT_DIR = ROOT / 'research_outputs' / 'fusion_training' / CFG.output_version
OUT_DIR.mkdir(parents=True, exist_ok=True)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
torch.manual_seed(CFG.random_state)
np.random.seed(CFG.random_state)

print('✓ Paths configured')
print('  FUSION_PATH:', FUSION_PATH)
print('  OUT_DIR:', OUT_DIR)
print('  device:', device)
print('  fold_selection_mode:', CFG.fold_selection_mode)
print('  max_folds:', CFG.max_folds)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✓ Paths configured
  FUSION_PATH: /content/drive/MyDrive/research_outputs/fusion/v1_fusion/fusion_dataset.csv
  OUT_DIR: /content/drive/MyDrive/research_outputs/fusion_training/v1_loso_two_tower
  device: cuda
  fold_selection_mode: fusion_representative
  max_folds: 5


In [8]:
# SECTION 3: Load and Split Feature Blocks
if not FUSION_PATH.exists():
    raise FileNotFoundError(FUSION_PATH)

df = pd.read_csv(FUSION_PATH)
label_col = 'label' if 'label' in df.columns else 'pseudo_label'
if label_col not in df.columns:
    raise ValueError('No label/pseudo_label column found.')

# Define feature groups explicitly to avoid duplicate EEG leakage.
all_cols = df.columns.tolist()
physio_prefixes = ('ECG__', 'EDA__', 'EMG__', 'RESP__')
physio_cols = [c for c in all_cols if c.startswith(physio_prefixes)]
eeg_cols = [c for c in all_cols if c.startswith('eeg_features_5s__')]

if not physio_cols or not eeg_cols:
    raise ValueError('Expected both physio and eeg feature columns are not available.')

subjects = df['subject_id'].astype(str).values
y_raw = df[label_col].astype(str).fillna('NA')
le = LabelEncoder()
y = le.fit_transform(y_raw)

# Keep only numeric-capable columns (some EEG metadata columns are strings like "filtered").
X_phys = df[physio_cols].apply(pd.to_numeric, errors='coerce')
X_eeg = df[eeg_cols].apply(pd.to_numeric, errors='coerce')

valid_phys_cols = [c for c in X_phys.columns if X_phys[c].notna().any()]
valid_eeg_cols = [c for c in X_eeg.columns if X_eeg[c].notna().any()]

X_phys = X_phys[valid_phys_cols].copy()
X_eeg = X_eeg[valid_eeg_cols].copy()
physio_cols = valid_phys_cols
eeg_cols = valid_eeg_cols

if X_phys.shape[1] == 0 or X_eeg.shape[1] == 0:
    raise ValueError('No numeric physio/eeg columns available after cleaning.')

print('✓ Feature blocks ready')
print('  X_phys:', X_phys.shape)
print('  X_eeg :', X_eeg.shape)
print('  classes:', list(le.classes_))
print('  subjects:', len(np.unique(subjects)))

✓ Feature blocks ready
  X_phys: (5640, 32)
  X_eeg : (5640, 97)
  classes: ['cognitive_load', 'high_stress', 'industrial_task', 'low_load', 'other']
  subjects: 52


In [4]:
# SECTION 4: Dataset + Model Definitions
class FusionDataset(Dataset):
    def __init__(self, Xp, Xe, y):
        self.Xp = torch.tensor(Xp, dtype=torch.float32)
        self.Xe = torch.tensor(Xe, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.long)

    def __len__(self):
        return len(self.y)

    def __getitem__(self, idx):
        return self.Xp[idx], self.Xe[idx], self.y[idx]

class TwoTowerFusion(nn.Module):
    def __init__(self, n_phys, n_eeg, n_classes, hidden=256, dropout=0.2):
        super().__init__()
        self.phys = nn.Sequential(
            nn.Linear(n_phys, hidden), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(hidden, hidden // 2), nn.ReLU()
        )
        self.eeg = nn.Sequential(
            nn.Linear(n_eeg, hidden), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(hidden, hidden // 2), nn.ReLU()
        )
        self.gate = nn.Sequential(
            nn.Linear(hidden, hidden // 2), nn.ReLU(), nn.Linear(hidden // 2, 2), nn.Sigmoid()
        )
        self.head = nn.Sequential(
            nn.Linear(hidden, hidden // 2), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(hidden // 2, n_classes)
        )

    def forward(self, xp, xe):
        hp = self.phys(xp)
        he = self.eeg(xe)
        cat = torch.cat([hp, he], dim=1)
        g = self.gate(cat)
        fused = torch.cat([g[:, 0:1] * hp, g[:, 1:2] * he], dim=1)
        return self.head(fused)

print('✓ Model classes ready')

✓ Model classes ready


In [9]:
# SECTION 5: LOSO Training Utilities (Run Fold-by-Fold)
from typing import Dict, List

def make_fold_preprocessed(X_train, X_test):
    imp = SimpleImputer(strategy='median')
    scl = StandardScaler()
    Xt = imp.fit_transform(X_train)
    Xt = scl.fit_transform(Xt)
    Xv = imp.transform(X_test)
    Xv = scl.transform(Xv)
    return Xt, Xv


def compute_class_weights(y_train, n_classes):
    counts = np.bincount(y_train, minlength=n_classes).astype(float)
    counts[counts == 0] = 1.0
    w = counts.sum() / (n_classes * counts)
    return torch.tensor(w, dtype=torch.float32, device=device)


def select_fusion_representative_subjects(df: pd.DataFrame, n_folds: int) -> List[str]:
    work = df[['subject_id', 'task_name', 'has_physio', 'has_eeg']].copy()
    work['subject_id'] = work['subject_id'].astype(str)
    work['task_name'] = work['task_name'].astype(str).str.lower()

    def family_flags(task_name: str) -> Dict[str, int]:
        return {
            'industrial': int(('cobot-task' in task_name) or ('manual-task' in task_name)),
            'cognitive': int(any(k in task_name for k in ['hanoi', 'n-back', 'stroop', 'mat'])),
            'stress': int('vr-plank' in task_name),
            'recovery': int(any(k in task_name for k in ['rest', 'meditation'])),
        }

    fam_df = work['task_name'].apply(family_flags).apply(pd.Series)
    prof = pd.concat([work[['subject_id', 'has_physio', 'has_eeg']], fam_df], axis=1)

    sub_prof = (
        prof.groupby('subject_id', as_index=False)
        .agg(
            has_physio=('has_physio', 'max'),
            has_eeg=('has_eeg', 'max'),
            industrial=('industrial', 'max'),
            cognitive=('cognitive', 'max'),
            stress=('stress', 'max'),
            recovery=('recovery', 'max'),
            n_rows=('subject_id', 'size'),
        )
    )

    sub_prof['family_coverage'] = sub_prof[['industrial', 'cognitive', 'stress', 'recovery']].sum(axis=1)
    sub_prof['multimodal_ready'] = ((sub_prof['has_physio'] > 0) & (sub_prof['has_eeg'] > 0)).astype(int)

    sub_prof = sub_prof.sort_values(
        ['multimodal_ready', 'family_coverage', 'n_rows', 'subject_id'],
        ascending=[False, False, False, True]
    ).reset_index(drop=True)

    chosen = sub_prof['subject_id'].head(n_folds).tolist()
    print('Selected representative subjects (fusion_representative):', chosen)
    return chosen


all_subjects = sorted(np.unique(subjects))
if CFG.max_folds > 0:
    if CFG.fold_selection_mode == 'fusion_representative':
        selected_subjects = select_fusion_representative_subjects(df, CFG.max_folds)
    else:
        selected_subjects = all_subjects[:CFG.max_folds]
else:
    selected_subjects = all_subjects

print(f'Prepared fold subjects ({len(selected_subjects)}): {selected_subjects}')

FOLD_CACHE_DIR = OUT_DIR / 'fold_cache'
FOLD_CACHE_DIR.mkdir(parents=True, exist_ok=True)
print('Fold cache dir:', FOLD_CACHE_DIR)


def run_one_fold(fold_index_1based: int):
    if fold_index_1based < 1 or fold_index_1based > len(selected_subjects):
        raise ValueError(f'fold_index must be in [1, {len(selected_subjects)}]')

    sid = selected_subjects[fold_index_1based - 1]
    test_mask = subjects == sid
    train_mask = ~test_mask

    Xp_tr, Xp_te = make_fold_preprocessed(X_phys.loc[train_mask], X_phys.loc[test_mask])
    Xe_tr, Xe_te = make_fold_preprocessed(X_eeg.loc[train_mask], X_eeg.loc[test_mask])

    y_tr = y[train_mask]
    y_te = y[test_mask]

    ds_tr = FusionDataset(Xp_tr, Xe_tr, y_tr)
    ds_te = FusionDataset(Xp_te, Xe_te, y_te)

    dl_tr = DataLoader(ds_tr, batch_size=CFG.batch_size, shuffle=True)
    dl_te = DataLoader(ds_te, batch_size=CFG.batch_size, shuffle=False)

    model = TwoTowerFusion(
        n_phys=Xp_tr.shape[1],
        n_eeg=Xe_tr.shape[1],
        n_classes=len(le.classes_),
        hidden=CFG.hidden_dim,
        dropout=CFG.dropout
    ).to(device)

    criterion = nn.CrossEntropyLoss(weight=compute_class_weights(y_tr, len(le.classes_)))
    optimizer = torch.optim.AdamW(model.parameters(), lr=CFG.lr, weight_decay=CFG.weight_decay)

    best_f1 = -1.0
    best_state = None
    no_improve = 0

    for epoch in range(1, CFG.max_epochs + 1):
        model.train()
        for xp_b, xe_b, y_b in dl_tr:
            xp_b, xe_b, y_b = xp_b.to(device), xe_b.to(device), y_b.to(device)
            optimizer.zero_grad()
            logits = model(xp_b, xe_b)
            loss = criterion(logits, y_b)
            loss.backward()
            optimizer.step()

        model.eval()
        preds = []
        trues = []
        with torch.no_grad():
            for xp_b, xe_b, y_b in dl_te:
                xp_b, xe_b = xp_b.to(device), xe_b.to(device)
                logits = model(xp_b, xe_b)
                y_hat = torch.argmax(logits, dim=1).cpu().numpy()
                preds.append(y_hat)
                trues.append(y_b.numpy())

        y_pred = np.concatenate(preds)
        y_true = np.concatenate(trues)
        f1m = f1_score(y_true, y_pred, average='macro')

        if f1m > best_f1:
            best_f1 = f1m
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            no_improve = 0
        else:
            no_improve += 1

        if no_improve >= CFG.patience:
            break

    model.load_state_dict(best_state)
    model.eval()
    preds = []
    trues = []
    with torch.no_grad():
        for xp_b, xe_b, y_b in dl_te:
            xp_b, xe_b = xp_b.to(device), xe_b.to(device)
            logits = model(xp_b, xe_b)
            y_hat = torch.argmax(logits, dim=1).cpu().numpy()
            preds.append(y_hat)
            trues.append(y_b.numpy())

    y_pred = np.concatenate(preds)
    y_true = np.concatenate(trues)

    fold_df_local = pd.DataFrame([{
        'fold': fold_index_1based,
        'test_subject': sid,
        'n_train': int(train_mask.sum()),
        'n_test': int(test_mask.sum()),
        'macro_f1': float(f1_score(y_true, y_pred, average='macro')),
        'balanced_acc': float(balanced_accuracy_score(y_true, y_pred)),
        'accuracy': float(accuracy_score(y_true, y_pred)),
        'best_val_macro_f1': float(best_f1)
    }])

    fold_path = FOLD_CACHE_DIR / f'fold_{fold_index_1based:02d}_metrics.csv'
    fold_df_local.to_csv(fold_path, index=False)

    print(f'✓ Completed fold {fold_index_1based}/{len(selected_subjects)} | sid={sid} | best_f1={best_f1:.4f}')
    print('  -', fold_path)
    display(fold_df_local)


print('Use the next cells to run folds individually (safe for Colab disconnects).')

Selected representative subjects (fusion_representative): ['10000', '1032', '1038', '1065', '1035']
Prepared fold subjects (5): ['10000', '1032', '1038', '1065', '1035']
Fold cache dir: /content/drive/MyDrive/research_outputs/fusion_training/v1_loso_two_tower/fold_cache
Use the next cells to run folds individually (safe for Colab disconnects).


## SECTION 5B: Run Folds Individually

Run each fold cell independently. Each completed fold writes a CSV in `fold_cache/`, so you can reconnect later and continue without losing prior folds.

In [10]:
# Fold 1
run_one_fold(1)

✓ Completed fold 1/5 | sid=10000 | best_f1=0.5752
  - /content/drive/MyDrive/research_outputs/fusion_training/v1_loso_two_tower/fold_cache/fold_01_metrics.csv


,fold,test_subject,n_train,n_test,macro_f1,balanced_acc,accuracy,best_val_macro_f1
0,1,10000,5477,163,0.575163,0.591596,0.711656,0.575163


In [13]:
# Fold 2
run_one_fold(2)

✓ Completed fold 2/5 | sid=1032 | best_f1=0.3914
  - /content/drive/MyDrive/research_outputs/fusion_training/v1_loso_two_tower/fold_cache/fold_02_metrics.csv


,fold,test_subject,n_train,n_test,macro_f1,balanced_acc,accuracy,best_val_macro_f1
0,2,1032,5477,163,0.391353,0.445418,0.601227,0.391353


In [14]:
# Fold 3
run_one_fold(3)

✓ Completed fold 3/5 | sid=1038 | best_f1=0.7545
  - /content/drive/MyDrive/research_outputs/fusion_training/v1_loso_two_tower/fold_cache/fold_03_metrics.csv


,fold,test_subject,n_train,n_test,macro_f1,balanced_acc,accuracy,best_val_macro_f1
0,3,1038,5480,160,0.754493,0.816137,0.8125,0.754493


In [15]:
# Fold 4
run_one_fold(4)

✓ Completed fold 4/5 | sid=1065 | best_f1=0.7715
  - /content/drive/MyDrive/research_outputs/fusion_training/v1_loso_two_tower/fold_cache/fold_04_metrics.csv


,fold,test_subject,n_train,n_test,macro_f1,balanced_acc,accuracy,best_val_macro_f1
0,4,1065,5482,158,0.771521,0.820371,0.822785,0.771521


In [16]:
# Fold 5
run_one_fold(5)

✓ Completed fold 5/5 | sid=1035 | best_f1=0.6007
  - /content/drive/MyDrive/research_outputs/fusion_training/v1_loso_two_tower/fold_cache/fold_05_metrics.csv


,fold,test_subject,n_train,n_test,macro_f1,balanced_acc,accuracy,best_val_macro_f1
0,5,1035,5483,157,0.600716,0.709717,0.464968,0.600716


In [17]:
# SECTION 6: Aggregate and Export

fold_files = sorted(FOLD_CACHE_DIR.glob('fold_*_metrics.csv')) if 'FOLD_CACHE_DIR' in globals() else []
if not fold_files:
    raise ValueError('No fold metrics found. Run at least one fold cell in SECTION 5B.')

fold_df = pd.concat([pd.read_csv(p) for p in fold_files], ignore_index=True)

agg = {
    'macro_f1_mean': float(fold_df['macro_f1'].mean()),
    'macro_f1_std': float(fold_df['macro_f1'].std()),
    'balanced_acc_mean': float(fold_df['balanced_acc'].mean()),
    'accuracy_mean': float(fold_df['accuracy'].mean()),
    'folds': int(len(fold_df)),
}

fold_path = OUT_DIR / 'loso_fold_metrics.csv'
agg_path = OUT_DIR / 'loso_aggregate_metrics.json'
manifest_path = OUT_DIR / 'run_manifest.json'

fold_df.to_csv(fold_path, index=False)
with open(agg_path, 'w') as f:
    json.dump(agg, f, indent=2)

manifest = {
    'config': asdict(CFG),
    'fusion_path': str(FUSION_PATH),
    'n_samples': int(len(df)),
    'n_phys_features': int(len(physio_cols)),
    'n_eeg_features': int(len(eeg_cols)),
    'label_classes': list(map(str, le.classes_)),
    'n_completed_folds': int(len(fold_files)),
    'completed_fold_files': [str(p) for p in fold_files],
    'outputs': {
        'fold_metrics': str(fold_path),
        'aggregate_metrics': str(agg_path)
    }
}
with open(manifest_path, 'w') as f:
    json.dump(manifest, f, indent=2)

print('✓ Exported two-tower outputs')
print('  completed folds:', len(fold_files))
print('  -', fold_path)
print('  -', agg_path)
print('  -', manifest_path)
print('Aggregate:', agg)

✓ Exported two-tower outputs
  completed folds: 5
  - /content/drive/MyDrive/research_outputs/fusion_training/v1_loso_two_tower/loso_fold_metrics.csv
  - /content/drive/MyDrive/research_outputs/fusion_training/v1_loso_two_tower/loso_aggregate_metrics.json
  - /content/drive/MyDrive/research_outputs/fusion_training/v1_loso_two_tower/run_manifest.json
Aggregate: {'macro_f1_mean': 0.6186489395270817, 'macro_f1_std': 0.15467666424149257, 'balanced_acc_mean': 0.676647671611109, 'accuracy_mean': 0.6826272797151293, 'folds': 5}


In [18]:
# Quick summary of current two-tower progress
import pandas as pd, json
from pathlib import Path

out_dir = OUT_DIR
agg_path = out_dir / 'loso_aggregate_metrics.json'
fold_path = out_dir / 'loso_fold_metrics.csv'

print('OUT_DIR:', out_dir)
if fold_path.exists():
    fold_df = pd.read_csv(fold_path)
    print('\nFold metrics:')
    display(fold_df)
else:
    print('Fold metrics not found yet.')

if agg_path.exists():
    with open(agg_path, 'r') as f:
        agg = json.load(f)
    print('\nAggregate metrics:')
    for k, v in agg.items():
        print(f'  {k}: {v}')
else:
    print('Aggregate metrics not found yet. Run SECTION 6 after at least one fold.')

OUT_DIR: /content/drive/MyDrive/research_outputs/fusion_training/v1_loso_two_tower

Fold metrics:


,fold,test_subject,n_train,n_test,macro_f1,balanced_acc,accuracy,best_val_macro_f1
0,1,10000,5477,163,0.575163,0.591596,0.711656,0.575163
1,2,1032,5477,163,0.391353,0.445418,0.601227,0.391353
2,3,1038,5480,160,0.754493,0.816137,0.812500,0.754493
3,4,1065,5482,158,0.771521,0.820371,0.822785,0.771521
4,5,1035,5483,157,0.600716,0.709717,0.464968,0.600716



Aggregate metrics:
  macro_f1_mean: 0.6186489395270817
  macro_f1_std: 0.15467666424149257
  balanced_acc_mean: 0.676647671611109
  accuracy_mean: 0.6826272797151293
  folds: 5
